# CodeSense Model Training & Evaluation

This notebook covers training a TF-IDF + Logistic Regression classification model on `error_dataset.csv`.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import re

def clean_text(value):
    text = "" if value is None else str(value)
    text = text.lower()
    text = re.sub(r"[^a-z0-9_+*/%=<>()\[\]{}.:,'\"\s\-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def combine_features(row):
    fields = ["code", "error_message", "error_type", "concept", "keywords"]
    return clean_text(" ".join(str(row.get(field, "")) for field in fields))

df = pd.read_csv("../dataset/error_dataset.csv").fillna("")
df["features"] = df.apply(combine_features, axis=1)

x_train, x_test, y_train, y_test = train_test_split(
    df["features"], df["error_category"], test_size=0.25, random_state=42, stratify=df["error_category"]
)

pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

pipeline.fit(x_train, y_train)
preds = pipeline.predict(x_test)
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))